# 02 — Train the multivariate forecaster

Trains the dependency-light ridge-rollout forecaster on the bundled sample store (or your real Zarr/npz export).
For the GPU path, open this in Colab and swap the import to the PyTorch LSTM in `models/forecast.py` + `models/train.py --torch`.

**Honesty note:** the bundled store is deterministic *synthetic* demo data (`meta.synthetic == True`).
Metrics computed on it describe the demo pipeline, not the real climate.

In [ ]:
import sys, json
sys.path.insert(0, '..')

import subprocess
subprocess.run([sys.executable, '../pipeline/run_pipeline.py', '--sample'], check=True)
subprocess.run([sys.executable, '../models/train.py'], check=True)

In [ ]:
from pipeline.export_store import load_store, VARIABLES

store = load_store('../data/samples/earth_pulse_sample.npz')
dates = list(store['dates'])
print('months:', len(dates), dates[0], '→', dates[-1])
print('cells:', len(store['grid_lat']), '× parameters:', len(VARIABLES))
print('parameters:', VARIABLES)
print('synthetic:', store['meta']['synthetic'], '| provenance:', store['meta']['provenance'])
print(json.dumps(store['meta'], indent=2)[:600])

# artifact written by models/train.py — the API prefers it over the linear baseline
import pathlib
artifact = pathlib.Path('../models/artifacts/forecaster.npz')
print('artifact exists:', artifact.exists(), '|', artifact.stat().st_size if artifact.exists() else 0, 'bytes')

## Rolling-origin backtest (what it means)

A forecast model must be judged on data it never trained on. **Rolling-origin** evaluation does this
honestly:

1. Pick a cut-off (`train_end = T - horizon`) — everything before it is "known".
2. Fit the model only on the prefix `[0, train_end)`.
3. Forecast the next `horizon` months.
4. Score the forecast against the held-out truth: MAE, RMSE, and **PicP** (the fraction of
   observations that fall inside the p10–p90 band — ideally ≈0.80).

Repeat across multiple cut-offs (a true rolling origin) to get a distribution of errors rather than
a single lucky split. `notebooks/03_backtest.ipynb` compares the ridge forecaster against the
transparent `linear_trend_seasonal` baseline so we never ship a complex model that loses to a line.

In [ ]:
from pipeline.build_features import prepare_cell_matrix
from models.backtest import backtest_cell

variables = ['temperature', 'precipitation', 'ndvi']
cell = 134
_, X = prepare_cell_matrix(store, cell, variables)

report = backtest_cell(X, variables, seq_len=24, test_months=24, horizon=24)
print(json.dumps(report, indent=2))